# SPIDER — Notebook 8B: new experiments requested by the IJIES reviewers

Uses exactly the cohort, crops and DINOv2 embeddings that Notebooks 1–4 saved on Drive,
and the same model code, so the new results are directly comparable with the paper.

| Section | Reviewer request | GPU time (T4, approx.) |
|---|---|---|
| 3. Learning curves, matched update budget | R1-3, R2-1 | 3 h |
| 4. Learning curves, early stopping on training-patient validation | R1-3, R2-1 | 3–4 h |
| 5. CNN on held-out scanners, two extra seeds | R2-4 | 0.5 h |
| 6. Learning curves, 8 epochs at every size | R1-3 | 1 h |
| 7. Training sensitivity: augmentation + early stopping, EfficientNet-B0 and ConvNeXt-Tiny | R2-3 | 4 h |

Every curve uses **3 outer partitions × 3 patient orderings** (the paper used 1 × 2), and the
geometry and DINOv2 arms are recomputed on the same partitions and orderings.

**How to run.** Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.
Every unit of work is saved to `MyDrive/spider/revision/B` as soon as it finishes.
If Colab disconnects or the GPU quota runs out, simply **Run all again later** — finished
units are skipped and the notebook continues where it stopped. You can also run the sections
on different days. When Section 8 reports *all units complete*, upload
`MyDrive/spider/revision/revision_B.zip` in the chat (Section 8 creates it; it contains only
predictions and tables, no images).

In [ ]:
# Cell 1 — setup, load the cohort, crops and embeddings exactly as Notebooks 3-4 did
import os, sys, json, time, math, glob, zipfile, platform, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

SMOKE = os.environ.get('SPIDER_SMOKE') == '1'          # local test mode only
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, IN_COLAB = '/content/drive/MyDrive', True
except ImportError:
    ROOT, IN_COLAB = os.environ.get('SPIDER_ROOT', '.'), False
DRIVE = f'{ROOT}/spider'
OUT = f'{DRIVE}/revision/B'
os.makedirs(OUT, exist_ok=True)
CROP_TAG = 'tight'

prim = pd.read_csv(f'{DRIVE}/spider_primary_cohort.csv')
z = np.load(f'{DRIVE}/spider_crops_{CROP_TAG}.npz', allow_pickle=True)
X_img, row_index = z['images'], z['row_index']
E = np.load(f'{DRIVE}/spider_dinov2_{CROP_TAG}.npy')
prim = prim.loc[row_index].reset_index(drop=True)
assert len(prim) == len(E) == len(X_img), "cohort / embeddings / crops misaligned"
seq_used = z['sequence'] if 'sequence' in z.files else np.array(['?'] * len(prim))

if not os.path.exists('overview.csv'):
    os.system('wget -q -O overview.csv "https://zenodo.org/records/10159290/files/overview.csv?download=1"')
ov = pd.read_csv('overview.csv')
ov['Patient'] = ov['new_file_name'].str.split('_').str[0].astype(int)
scan = (ov.groupby('Patient')
          .agg(scanner=('ManufacturerModelName', lambda s: s.mode().iat[0])).reset_index())
prim = prim.merge(scan, on='Patient', how='left')

INV = ['disc_height_ratio', 'ap_ratio', 'canal_ratio', 'volume_ratio', 'slip_pct']
TARGETS = {'disc narrowing': 'narrowing', 'disc bulging': 'bulging',
           'Pfirrmann >= 4': 'pfirrmann_severe', 'Modic (any)': 'modic_any'}
levels, pats = prim['ivd_label'].values, prim['Patient'].values

import torch, torchvision
versions = {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__,
            'torch': torch.__version__, 'torchvision': torchvision.__version__,
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}
import sklearn; versions['sklearn'] = sklearn.__version__
json.dump(versions, open(f'{OUT}/versions.json', 'w'), indent=1)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
if not SMOKE:
    assert dev == 'cuda', "Runtime > Change runtime type > T4 GPU, then Run all again"
print(f"{len(prim)} discs, {prim.Patient.nunique()} patients; sequences {pd.Series(seq_used).value_counts().to_dict()}")
print(versions)

pd.DataFrame({'row': np.arange(len(prim)), 'Patient': pats, 'ivd_label': levels,
              'scanner': prim.scanner.values, 'sequence': seq_used,
              **{v: prim[v].astype(int).values for v in TARGETS.values()}}
             ).to_csv(f'{OUT}/cohort_rows.csv', index=False)


## Cell 2 — shared code (identical to Notebooks 3–4, plus the new CNN training protocols)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import torch.nn as nn, torch.nn.functional as F

SEED = 0
FRACTIONS = [0.05, 0.10, 0.25, 0.50, 1.00]
PARTITIONS = [0, 1, 2]           # partition 0 = the split used in the paper
DRAWS = [0, 1, 2]                # draws 0 and 1 of partition 0 = the orderings used in the paper
CNN_TARGETS = ['narrowing', 'pfirrmann_severe']
BATCH, EPOCHS = 32, 8
MAX_EPOCHS, PATIENCE, MIN_EPOCHS = 30, 5, 3
if SMOKE:
    FRACTIONS, PARTITIONS, DRAWS, CNN_TARGETS = [0.5, 1.0], [0], [0], ['narrowing']
    EPOCHS, MAX_EPOCHS, PATIENCE, MIN_EPOCHS = 1, 2, 1, 1


def outer_splits(y, part):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=part)
    return list(sgkf.split(np.zeros(len(y)), y, pats))


def subsample(tr, frac, draw, fold, part):
    # nested patient subsets; part 0 reproduces Notebook 4 exactly
    up = np.unique(pats[tr])
    order = np.random.default_rng(1_000_000 * part + 10_000 * draw + fold).permutation(up)
    keep = order[:max(2, math.ceil(frac * len(up)))]
    return tr[np.isin(pats[tr], keep)]


def _zscore(ref, ref_lvl, X, lvl, min_per_level):
    out = X.copy()
    gm, gs = ref.mean(), ref.std().replace(0, 1)
    for l in np.unique(lvl):
        m = ref_lvl == l
        ok = m.sum() >= min_per_level
        mu = ref[m].mean() if ok else gm
        sd = ref[m].std().replace(0, 1) if ok else gs
        out.loc[lvl == l] = (X.loc[lvl == l] - mu) / sd
    return out.fillna(0) if min_per_level == 8 else out


def _lr(seed=SEED):
    return LogisticRegression(max_iter=5000, class_weight='balanced', random_state=seed)


def fit_geometry(tr, te, y, min_per_level=8):
    A, B = prim[INV].iloc[tr], prim[INV].iloc[te]
    A2 = _zscore(A, levels[tr], A, levels[tr], min_per_level)
    B2 = _zscore(A, levels[tr], B, levels[te], min_per_level)
    m = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), _lr()).fit(A2, y[tr])
    return m.predict_proba(B2)[:, 1]


def fit_dinov2(tr, te, y):
    k = max(2, min(128, len(tr) - 1))
    m = make_pipeline(StandardScaler(), PCA(n_components=k, random_state=SEED), _lr()).fit(E[tr], y[tr])
    return m.predict_proba(E[te])[:, 1]


def fit_fusion(tr, te, y):           # Notebook 3, min 20 discs per level
    A, B = prim[INV].iloc[tr], prim[INV].iloc[te]
    A2 = _zscore(A, levels[tr], A, levels[tr], 20).values
    B2 = _zscore(A, levels[tr], B, levels[te], 20).values
    pca = make_pipeline(StandardScaler(), PCA(n_components=min(32, len(tr) - 1), random_state=SEED)).fit(E[tr])
    Atr, Ate = np.hstack([A2, pca.transform(E[tr])]), np.hstack([B2, pca.transform(E[te])])
    w = np.where(y[tr] == 1, (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1), 1.)
    g = HistGradientBoostingClassifier(max_depth=3, max_iter=300, random_state=SEED)
    g.fit(Atr, y[tr], sample_weight=w)
    return g.predict_proba(Ate)[:, 1]


# ---------------- CNN ----------------
Xt = torch.from_numpy(X_img).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)   # as in Notebooks 3-4
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def make_net(arch):
    w = None if SMOKE else 'IMAGENET1K_V1'
    if arch == 'effb0':
        net = torchvision.models.efficientnet_b0(weights=w)
        net.classifier[1] = nn.Linear(net.classifier[1].in_features, 1)
    elif arch == 'convnext_t':
        net = torchvision.models.convnext_tiny(weights=w)
        net.classifier[2] = nn.Linear(net.classifier[2].in_features, 1)
    else:
        raise ValueError(arch)
    return net.to(dev)


def prep(xb, arch):
    xb = xb.to(dev)
    if arch == 'convnext_t':                 # standard ImageNet normalisation for ConvNeXt
        xb = (xb - IMNET_MEAN.to(dev)) / IMNET_STD.to(dev)
    return xb


def augment(xb, gen):
    # per-image random affine (rotation +-10 deg, translation +-5%, scale 0.9-1.1) and
    # brightness/contrast jitter (+-10%); no flips, which would change anatomical orientation
    n = xb.shape[0]
    r = lambda lo, hi: lo + (hi - lo) * torch.rand(n, generator=gen)
    ang, sc = r(-10, 10) * math.pi / 180, r(0.9, 1.1)
    tx, ty = r(-0.05, 0.05), r(-0.05, 0.05)
    th = torch.zeros(n, 2, 3)
    th[:, 0, 0], th[:, 0, 1], th[:, 0, 2] = torch.cos(ang) / sc, -torch.sin(ang) / sc, tx
    th[:, 1, 0], th[:, 1, 1], th[:, 1, 2] = torch.sin(ang) / sc, torch.cos(ang) / sc, ty
    grid = F.affine_grid(th.to(xb.device), xb.shape, align_corners=False)
    xb = F.grid_sample(xb, grid, padding_mode='border', align_corners=False)
    b, c = r(0.9, 1.1).to(xb.device).view(n, 1, 1, 1), r(0.9, 1.1).to(xb.device).view(n, 1, 1, 1)
    m = xb.mean(dim=(1, 2, 3), keepdim=True)
    return ((xb - m) * c + m * b).clamp(0, 1)


def infer(net, idx, arch):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(idx), 64):
            out.append(torch.sigmoid(net(prep(Xt[idx[i:i + 64]], arch)).squeeze(1)).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


def train_cnn(tr, te, y, seed, scheme, total_updates=None, arch='effb0', aug=False):
    """scheme: 'epochs8'         8 epochs at any training size, as the primary CNN (Notebook 3)
               'matched_updates' exactly total_updates optimiser steps, cosine schedule
               'early_stop'      inner validation split of TRAINING patients (about 1/6);
                                 up to 30 epochs, stop after 5 epochs without lower validation
                                 loss, keep the best epoch; constant learning rate
       returns test probs, train probs, info dict"""
    torch.manual_seed(seed); np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    gen = torch.Generator().manual_seed(seed)
    fit_idx, val_idx = tr, np.array([], int)
    if scheme == 'early_stop':
        n_pat = len(np.unique(pats[tr]))
        k = 6 if n_pat >= 12 else max(2, n_pat // 2)
        try:
            sg = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
            a, b = next(sg.split(np.zeros(len(tr)), y[tr], pats[tr]))
        except ValueError:
            from sklearn.model_selection import GroupKFold
            a, b = next(GroupKFold(n_splits=k).split(np.zeros(len(tr)), y[tr], pats[tr]))
        fit_idx, val_idx = tr[a], tr[b]
    net = make_net(arch)
    lr = 1e-4 if arch == 'convnext_t' else 3e-4
    wd = 0.05 if arch == 'convnext_t' else 1e-4
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    pw = torch.tensor([(y[fit_idx] == 0).sum() / max((y[fit_idx] == 1).sum(), 1)], dtype=torch.float32).to(dev)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    ytr = torch.tensor(y).float()
    steps_per_epoch = math.ceil(len(fit_idx) / BATCH)
    if scheme == 'epochs8':
        total = EPOCHS * steps_per_epoch
    elif scheme == 'matched_updates':
        total = int(total_updates)
    else:
        total = MAX_EPOCHS * steps_per_epoch
    # epochs8 reproduces Notebook 3 exactly: cosine schedule stepped once per epoch (T_max = 8 epochs);
    # matched_updates reproduces Notebook 4: cosine schedule stepped once per update
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total) if scheme == 'matched_updates'
             else torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS) if scheme == 'epochs8'
             else None)
    step, epoch, best, best_state, best_epoch, bad = 0, 0, np.inf, None, 0, 0
    while step < total:
        net.train()
        perm = fit_idx[torch.randperm(len(fit_idx), generator=gen).numpy()]
        for i in range(0, len(perm), BATCH):
            bi = perm[i:i + BATCH]
            xb = Xt[bi]
            if aug:
                xb = augment(xb, gen)
            opt.zero_grad()
            lossf(net(prep(xb, arch)).squeeze(1), ytr[bi].to(dev)).backward()
            opt.step(); step += 1
            if scheme == 'matched_updates':
                sched.step()
            if step >= total:
                break
        epoch += 1
        if scheme == 'epochs8':
            sched.step()
        if scheme == 'early_stop' and len(val_idx):
            pv = infer(net, val_idx, arch).clip(1e-6, 1 - 1e-6)
            yv = y[val_idx]
            vloss = float(-np.mean(pw.item() * yv * np.log(pv) + (1 - yv) * np.log(1 - pv)))
            if vloss < best - 1e-4:
                best, bad, best_epoch = vloss, 0, epoch
                best_state = {k_: v.detach().cpu().clone() for k_, v in net.state_dict().items()}
            else:
                bad += 1
            if epoch >= MIN_EPOCHS and bad >= PATIENCE:
                break
    if best_state is not None:
        net.load_state_dict(best_state)
    pte, ptr = infer(net, te, arch), infer(net, tr, arch)
    info = dict(updates=step, epochs_run=epoch, best_epoch=best_epoch, n_fit=len(fit_idx),
                n_val=len(val_idx), steps_per_epoch=steps_per_epoch)
    del net; torch.cuda.empty_cache() if dev == 'cuda' else None
    return pte, ptr, info


UNITS_LOG = f'{OUT}/units_log.csv'
def log_unit(**kw):
    pd.DataFrame([kw]).to_csv(UNITS_LOG, mode='a', header=not os.path.exists(UNITS_LOG), index=False)

print("shared code ready")


## Cell 3 — geometry and DINOv2 curves on all partitions and orderings (CPU part, ~5 min)

The 100% point is the same for every ordering, so it is computed once per partition.

In [ ]:
LC = f'{OUT}/lc'; os.makedirs(LC, exist_ok=True)
t0 = time.time()
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    for part in PARTITIONS:
        f = f'{LC}/geodino_{tcol}_p{part}.npz'
        if os.path.exists(f):
            continue
        splits = outer_splits(y, part)
        res = {}
        for fi, frac in enumerate(FRACTIONS):
            for draw in (DRAWS if frac < 1 else [0]):
                for arm, fit in (('geometry', fit_geometry), ('dinov2', fit_dinov2)):
                    p = np.full(len(y), np.nan); info = []
                    for k, (tr, te) in enumerate(splits):
                        sub = subsample(tr, frac, draw, k, part)
                        if len(np.unique(y[sub])) < 2:
                            continue
                        p[te] = fit(sub, te, y)
                        info.append((len(sub), int(y[sub].sum()), len(np.unique(pats[sub])),
                                     len(np.unique(pats[sub][y[sub] == 1]))))
                    res[f'{arm}|{frac}|{draw}'] = p
                    res[f'info|{arm}|{frac}|{draw}'] = np.array(info)
        res['folds'] = np.concatenate([np.full(len(te), k) for k, (tr, te) in enumerate(splits)])
        res['fold_rows'] = np.concatenate([te for tr, te in splits])
        np.savez(f, **res)
    print(f"  {tname} done ({time.time()-t0:.0f}s)", flush=True)
print("geometry and DINOv2 curves complete")


## Cells 4–6 — CNN learning curves under three training protocols

* **matched_updates**: every training size gets exactly the number of optimiser steps that the
  full-size CNN gets in that fold (8 epochs of the full training fold), so sample size is varied
  while optimisation exposure is held constant.
* **early_stop**: a convergence protocol — validation on held-back *training* patients only.
* **epochs8**: 8 epochs at every size, as the primary CNN (fewer steps for smaller sets).

Each unit (target × protocol × partition × fraction × ordering) is checkpointed.

In [ ]:
SCHEME_ID = {'matched_updates': 1, 'early_stop': 2, 'epochs8': 3}

def run_cnn_curves(scheme):
    t_all = time.time()
    todo = [(tcol, part, fi, frac, draw) for tcol in CNN_TARGETS for part in PARTITIONS
            for fi, frac in enumerate(FRACTIONS) for draw in (DRAWS if frac < 1 else [0])]
    for n_done, (tcol, part, fi, frac, draw) in enumerate(todo, 1):
        f = f'{LC}/cnn_{scheme}_{tcol}_p{part}_f{int(frac*100):03d}_d{draw}.npz'
        if os.path.exists(f):
            continue
        y = prim[tcol].astype(int).values
        splits = outer_splits(y, part)
        p = np.full(len(y), np.nan); infos = []
        t0 = time.time()
        for k, (tr, te) in enumerate(splits):
            sub = subsample(tr, frac, draw, k, part)
            if len(np.unique(y[sub])) < 2:
                continue
            seed = 100_000 * SCHEME_ID[scheme] + 10_000 * part + 1_000 * draw + 100 * fi + k
            full_updates = EPOCHS * math.ceil(len(tr) / BATCH)
            pte, _, info = train_cnn(sub, te, y, seed, scheme, total_updates=full_updates)
            p[te] = pte
            info.update(fold=k, seed=seed, n_train=len(sub), pos_discs=int(y[sub].sum()),
                        patients=len(np.unique(pats[sub])),
                        pos_patients=len(np.unique(pats[sub][y[sub] == 1])))
            infos.append(info)
        np.savez(f, p=p, info=json.dumps(infos))
        auc = roc_auc_score(y[~np.isnan(p)], p[~np.isnan(p)])
        log_unit(section='lc', scheme=scheme, target=tcol, part=part, frac=frac, draw=draw,
                 auc=round(auc, 4), seconds=round(time.time() - t0),
                 updates=';'.join(str(i['updates']) for i in infos))
        el = time.time() - t_all
        print(f"  {scheme:15s} {tcol:16s} p{part} {frac:4.0%} d{draw}  AUC {auc:.3f}  "
              f"[{n_done}/{len(todo)}, {el/60:.0f} min elapsed]", flush=True)
    print(f"{scheme}: all units complete")

run_cnn_curves('matched_updates')


In [ ]:
run_cnn_curves('early_stop')

## Cell 7 — CNN on held-out scanners: two more seeds (seed 7 from Notebook 3 is reused if present)

In [ ]:
LS = f'{OUT}/loso'; os.makedirs(LS, exist_ok=True)
counts = prim.drop_duplicates('Patient').scanner.value_counts()
held_out = [s for s, n in counts.items() if n >= (15 if not SMOKE else 1)]
print("held-out scanners:", held_out)
t0 = time.time()
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    for s in held_out:
        te = np.flatnonzero((prim.scanner == s).values)
        tr = np.flatnonzero((prim.scanner != s).values)
        if len(np.unique(y[te])) < 2:
            continue
        f = f'{LS}/descriptor_arms_{tcol}_{s}.npz'
        if not os.path.exists(f):            # deterministic arms, saved for patient-level intervals
            np.savez(f, te=te, geometry=fit_geometry(tr, te, y, 20), dinov2=fit_dinov2(tr, te, y),
                     fusion=fit_fusion(tr, te, y))
        for seed in (7, 11, 13):
            f = f'{LS}/cnn_{tcol}_{s}_seed{seed}.npz'
            if os.path.exists(f):
                continue
            old = f'{DRIVE}/nb3_cnn/loso_{tcol}_{s}.npz'
            if seed == 7 and os.path.exists(old):
                np.savez(f, te=te, p=np.load(old)['p'], reused=1)
                continue
            pte, _, info = train_cnn(tr, te, y, seed, 'epochs8')
            np.savez(f, te=te, p=pte, reused=0, info=json.dumps(info))
            log_unit(section='loso', scheme='epochs8', target=tcol, part=s, frac=1, draw=seed,
                     auc=round(roc_auc_score(y[te], pte), 4), seconds=round(time.time() - t0),
                     updates=info['updates'])
            print(f"  {tcol:16s} {s:12s} seed {seed}  AUC {roc_auc_score(y[te], pte):.3f}"
                  f"  ({(time.time()-t0)/60:.0f} min)", flush=True)
print("held-out scanner runs complete")


## Cell 8 — learning curves with 8 epochs at every size

In [ ]:
run_cnn_curves('epochs8')

## Cell 9 — training sensitivity and a stronger comparator (R2-3)

Same five-fold patient-grouped splits as the paper's repeated cross-validation (repeats 0–2),
all four targets. Two protocols, both with augmentation and early stopping on training-patient
validation: EfficientNet-B0 (the paper's architecture) and ConvNeXt-Tiny (a stronger ImageNet
backbone, 28 M parameters).

In [ ]:
SN = f'{OUT}/sensitivity'; os.makedirs(SN, exist_ok=True)
SENS_REPEATS = [0, 1, 2] if not SMOKE else [0]
from sklearn.metrics import roc_curve
t0 = time.time()
for arch in (['effb0', 'convnext_t'] if not SMOKE else ['effb0']):
    for tname, tcol in TARGETS.items():
        if SMOKE and tcol != 'narrowing':
            continue
        y = prim[tcol].astype(int).values
        for rep in SENS_REPEATS:
            f = f'{SN}/{arch}_aug_es_{tcol}_rep{rep}.npz'
            if os.path.exists(f):
                continue
            sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=rep)   # as Notebook 3
            p = np.zeros(len(y)); b = np.zeros(len(y), int); infos = []
            for k, (tr, te) in enumerate(sgkf.split(np.zeros(len(y)), y, pats)):
                seed = 500_000 + (0 if arch == 'effb0' else 50_000) + 1000 * rep + k
                pte, ptr, info = train_cnn(tr, te, y, seed, 'early_stop', arch=arch, aug=True)
                fpr, tpr, thr = roc_curve(y[tr], ptr)
                p[te], b[te] = pte, (pte >= thr[np.argmax(tpr - fpr)]).astype(int)
                info.update(fold=k, seed=seed); infos.append(info)
            np.savez(f, p=p, b=b, info=json.dumps(infos))
            log_unit(section='sens', scheme=f'{arch}_aug_es', target=tcol, part=rep, frac=1, draw=0,
                     auc=round(roc_auc_score(y, p), 4), seconds=round(time.time() - t0),
                     updates=';'.join(str(i['updates']) for i in infos))
            print(f"  {arch:10s} {tcol:16s} rep {rep}  AUC {roc_auc_score(y, p):.3f}"
                  f"  ({(time.time()-t0)/60:.0f} min)", flush=True)
print("sensitivity runs complete")


## Cell 10 — progress check and zip for upload

In [ ]:
expected = {
  'lc_geodino': len(TARGETS) * len(PARTITIONS),
  **{f'lc_{s}': len(CNN_TARGETS) * len(PARTITIONS) * (len(DRAWS) * (len(FRACTIONS) - 1) + 1)
     for s in ['matched_updates', 'early_stop', 'epochs8']},
}
have = {
  'lc_geodino': len(glob.glob(f'{OUT}/lc/geodino_*.npz')),
  **{f'lc_{s}': len(glob.glob(f'{OUT}/lc/cnn_{s}_*.npz')) for s in ['matched_updates', 'early_stop', 'epochs8']},
  'loso_cnn': len(glob.glob(f'{OUT}/loso/cnn_*.npz')),
  'sensitivity': len(glob.glob(f'{OUT}/sensitivity/*.npz')),
}
for k, v in have.items():
    print(f"  {k:22s} {v:4d}" + (f" / {expected[k]}" if k in expected else ''))
done = all(have[k] >= expected[k] for k in expected)
print("all learning-curve units complete" if done else "some units still missing - Run all again to continue")

zpath = f'{DRIVE}/revision/revision_B.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
    for dp, dn, fn in os.walk(OUT):
        for fname in fn:
            p_ = os.path.join(dp, fname)
            zf.write(p_, os.path.relpath(p_, f'{DRIVE}/revision'))
print(f"saved {zpath} ({os.path.getsize(zpath)/1e6:.1f} MB) - upload this file in the chat")
if IN_COLAB:
    from google.colab import files
    files.download(zpath)
